In [0]:
%run ./02-Stream-JSON-Processing

In [0]:
class invoiceTestSuit:
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/landing/datasets/json_files/'
        self.test_data_path='/Volumes/cat_spark_streaming/jsonstream/dataset/json-datasets/'
        self.checkpoint_path='/Volumes/cat_spark_streaming/jsonstream/checkpoint_dir/'
        self.catalog_name='cat_spark_streaming'
        self.schema_name='jsonstream'
        self.table_name='invoice'

    def cleanTest(self):
        dbutils.fs.rm(self.test_data_path,True)
        dbutils.fs.rm(self.checkpoint_path,True)
        print('Cleaned and Recreating directories: ')
        dbutils.fs.mkdirs(self.test_data_path)
        dbutils.fs.mkdirs(self.checkpoint_path)
        spark.sql(f"drop table {self.catalog_name}.{self.schema_name}.{self.table_name}")

    def ingestFiles(self,itr):
        print(f"\tStarting Ingestion...", end= '') 
        dbutils.fs.cp(f'{self.data_path}invoices-{itr}.json', f'{self.test_data_path}')
        print('Completed')

    def assertResult(self,expected_count):
        actual_count=spark.sql(f'select count(*) from {self.catalog_name}.{self.schema_name}.{self.table_name}').collect()[0][0]
        assert actual_count==expected_count, f"\tTest failed! {actual_count} is not equal to {expected_count}"

    def waitForMicrobatch(self,sleep=30):
        import time
        print(f"\tWait for {sleep} seconds",end='')
        time.sleep(sleep)
        print("Done")

    def runTests(self):
        isobj=invoiceStream()
        self.cleanTest()

        print("Starting the first iteration:")
        self.ingestFiles(1)
        isobj.stream_processing()
        self.waitForMicrobatch()
        self.assertResult(1253)
        print("Validation passed.\n")


        print("Starting the second iteration:")
        self.ingestFiles(2)
        isobj.stream_processing()
        self.waitForMicrobatch()
        self.assertResult(2510)
        print("Validation passed.\n")


        print("Starting the third iteration:")
        self.ingestFiles(3)
        isobj.stream_processing()
        self.waitForMicrobatch()
        self.assertResult(3994)       
        print("Validation passed.\n")

In [0]:
itsobj = invoiceTestSuit()
itsobj.runTests()